# 09 - PySpark Rolling Trend and Peak


## Setup
Target: Configure Spark + JDBC helper.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
spark = (SparkSession.builder.appName('capacity-demo').config('spark.driver.host','127.0.0.1').config('spark.driver.bindAddress','127.0.0.1').config('spark.jars.packages','org.postgresql:postgresql:42.7.4').getOrCreate())
JDBC_URL = 'jdbc:postgresql://host.docker.internal:5432/observability'
def load_query(q: str):
    return (spark.read.format('jdbc').option('url', JDBC_URL).option('dbtable', f'({q}) t').option('user','obs_user').option('password','obs_pass').option('driver','org.postgresql.Driver').load())


## Rolling Features
Target: Compute rolling avg and peak with Spark windows.


In [ ]:
q = """SELECT sampled_at, host, cpu_pct, region AS application, env AS service FROM lab.telemetry_cpu_raw WHERE sampled_at >= now() - interval '14 days'"""
df = load_query(q)
h = df.withColumn('hour_bucket', F.date_trunc('hour', 'sampled_at')).groupBy('host','application','service','hour_bucket').agg(F.avg('cpu_pct').alias('avg_cpu'), F.max('cpu_pct').alias('peak_cpu'))
w = Window.partitionBy('host','application','service').orderBy('hour_bucket').rowsBetween(-23, 0)
out = h.withColumn('cpu_24h_rolling_avg', F.round(F.avg('avg_cpu').over(w),2)).withColumn('cpu_24h_rolling_peak', F.round(F.max('peak_cpu').over(w),2)).orderBy('host','application','service','hour_bucket')
out.show(40, truncate=False)
